# Practical 9: Named Entity Recognition (NER) with spaCy

**Concept:** **Named Entity Recognition (NER)** finds and classifies **named entities** in text into predefined categories such as `PERSON`, `ORG` (organisation), `GPE` (city/country), `DATE`, `MONEY`, etc. It is a key building block for information extraction, search and question answering.

In this practical we use `spaCy`, a fast industrial-strength NLP library, to extract and **visualize entities** with `displacy`.


## Install spaCy


In [4]:
%pip install spacy



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /home/tejas/.local/share/pipx/venvs/notebook/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Download the English Model
spaCy needs a trained pipeline to do NER. The small English model (`en_core_web_sm`) already knows how to detect entities out of the box.


In [11]:
%python3 -m spacy download en_core_web_sm


UsageError: Line magic function `%python3` not found (But cell magic `%%python3` exists, did you mean that instead?).


## Import and Load the Model


In [ ]:
import spacy
from spacy import displacy

from collections import Counter

# Load the pretrained English pipeline
nlp = spacy.load("en_core_web_sm")

print("Model loaded:", spacy.util.get_package_path("en_core_web_sm").name if hasattr(spacy.util.get_package_path("en_core_web_sm"), "name") else "en_core_web_sm")


## Sample Text
We use a news-style paragraph full of people, organisations, places, dates and money — ideal for NER.


In [ ]:
text = (
    "Apple Inc. announced on Monday that CEO Tim Cook will visit Mumbai in March "
    "to open a new store that cost $25 million to build. The company plans to hire "
    "500 engineers in Bangalore next year. Microsoft and Google are also expanding "
    "their offices in India."
)

print(text)


## Process the Text
`nlp(text)` returns a `Doc` object that already contains the detected entities.


In [ ]:
doc = nlp(text)

print("Sentences")
print("-" * 50)
for i, sent in enumerate(doc.sents, start=1):
    print(f"{i}. {sent}")


## Extract Named Entities
Each entity gives us its **text**, **label**, **start** and **end** character offsets. `spacy.explain` decodes the label into plain English.


In [ ]:
print("Named Entities")
print("-" * 50)
for ent in doc.ents:
    print(f"{ent.text:<20} {ent.label_:<10} start={ent.start:<4} end={ent.end:<4} {spacy.explain(ent.label_)}")


## Entity Type Summary
Count how many entities belong to each type.


In [ ]:
labels = Counter(ent.label_ for ent in doc.ents)

print("Entity Type Counts")
print("-" * 50)
for label, count in labels.most_common():
    print(f"{label:<10} {count}  ({spacy.explain(label)})")


## Visualizing Entities with displaCy
`displacy` renders the entities as color-coded spans. In a Jupyter notebook we embed the generated HTML with `IPython.display`.


In [ ]:
from IPython.display import HTML, display

html = displacy.render(doc, style="ent", jupyter=False)
display(HTML(html))


## Filtering Which Entities to Show
We can ask displaCy to show only selected entity types — here only `PERSON`, `ORG` and `GPE`.


In [ ]:
options = {"ents": ["PERSON", "ORG", "GPE"], "colors": {"PERSON": "#f6b26b", "ORG": "#8fd9a8", "GPE": "#9fc5e8"}}

html = displacy.render(doc, style="ent", jupyter=False, options=options)
display(HTML(html))


## Entity Extraction as Structured Data
For a real application we usually want the entities as a clean list/tabular output, not just a rendering. We also show the entity `iob` tag (B/I/O: beginning / inside / outside an entity).


In [ ]:
rows = []
for ent in doc.ents:
    rows.append({"text": ent.text, "label": ent.label_, "start": ent.start, "end": ent.end})

print("Entity Table")
print("-" * 50)
for row in rows:
    print(f"{row['text']:<20} {row['label']:<10} chars {row['start']}-{row['end']}")

print()
print("Per-token IOB tags")
print("-" * 50)
for token in doc:
    print(f"{token.text:<12} {token.ent_iob_} {token.ent_type_ or ''}")


## Demo on Another Text
Let's run the same model on a different paragraph to confirm it generalizes.


In [ ]:
text2 = (
    "On June 15, 2020, Sundar Pichai, the CEO of Alphabet, met Prime Minister "
    "Narendra Modi in New Delhi to discuss a $10 billion investment in technology."
)

doc2 = nlp(text2)

print("Named Entities")
print("-" * 50)
for ent in doc2.ents:
    print(f"{ent.text:<25} {ent.label_:<8} {spacy.explain(ent.label_)}")

print()
html = displacy.render(doc2, style="ent", jupyter=False)
display(HTML(html))


## Summary

In this practical we:
1. Loaded spaCy's pretrained English pipeline `en_core_web_sm`.
2. Extracted **named entities** (PERSON, ORG, GPE, DATE, MONEY, CARDINAL...) from text.
3. Summarized entity types and decoded labels with `spacy.explain`.
4. **Visualized entities** using `displaCy` in the notebook.
5. Viewed entities as structured data and as per-token IOB tags.

spaCy runs NER out-of-the-box with zero training — a great example of using pretrained NLP models.
